# Star Cluster Injection Pipeline - RSP

This notebook follows the methodology from the official Rubin DP0.2 tutorial:
> **[DP02_14 - Injecting Synthetic Sources](https://github.com/rubin-dp0/tutorial-notebooks/blob/main/DP02_14_Injecting_Synthetic_Sources.ipynb)**

Key differences from the tutorial (which injects individual stars):
- We inject **extended sources** (star clusters) using smooth 2D profiles
- We support multiple spatial profiles: Plummer, King, EFF, Sersic
- We add multi-band injection and completeness analysis

Clone: `git clone https://github.com/whosneha/INJECT.git`

---
## 1. Setup and Imports

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm, AsinhNorm
from matplotlib.patches import Circle
from datetime import datetime

USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/INJECT/star-cluster-injection-pipeline'

if os.path.exists(PIPELINE_PATH):
    print(f'✓ Pipeline found at: {PIPELINE_PATH}')
    sys.path.insert(0, PIPELINE_PATH)
else:
    print(f'✗ Pipeline NOT found. Run: cd ~ && git clone https://github.com/whosneha/INJECT.git')

In [ ]:
# LSST stack imports
from lsst.daf.butler import Butler
from lsst.geom import Point2D

from src.inject import inject_cluster, create_injection_catalog, inject_from_catalog
from src.inject import inject_multiband, create_multiband_catalog
from src.light_profiles import PlummerProfile, KingProfile, mag_to_flux
from src.cluster_models import create_cluster, DiscreteStarCluster
from src.psf_convolution import (
    get_psf_from_coadd,
    get_psf_fwhm_from_coadd,
    get_psf_size_from_coadd,
    convolve_with_psf,
    sample_psf_across_image
)
from src.visualization import plot_postage_stamps, plot_stamp_grid, plot_injection_summary
from src.retrieval import ClusterRetrieval, run_source_detection

print('✓ All imports successful!')

---
## 2. Load Coadd Image

Following the Rubin tutorial approach - load a `deepCoadd` using the Butler.
Reference: [DP02_14 Section 2](https://github.com/rubin-dp0/tutorial-notebooks/blob/main/DP02_14_Injecting_Synthetic_Sources.ipynb)

In [ ]:
REPO       = 's3://butler-us-central1-dp02'
COLLECTION = '2.2i/runs/DP0.2'
DATA_ID    = {'tract': 4431, 'patch': 17, 'band': 'i'}

butler_configs = [
    ('s3://butler-us-central1-dp02', '2.2i/runs/DP0.2'),
    ('/repo/dc2',                    '2.2i/runs/DP0.2'),
]

butler = None
for repo, collection in butler_configs:
    try:
        print(f'Trying: {repo}')
        butler = Butler(repo, collections=collection)
        REPO, COLLECTION = repo, collection
        print('✓ Success!')
        break
    except Exception as e:
        print(f'  Failed: {str(e)[:80]}')

if butler is None:
    print('ERROR: Could not connect. Uncomment mock image cell below.')

In [ ]:
# Load coadd and extract image + photometric calibration
if butler is not None:
    exposure = butler.get('deepCoadd', dataId=DATA_ID)
    image    = exposure.image.array.copy()

    try:
        photo_calib = exposure.getPhotoCalib()
        ZEROPOINT   = 31.4  # Rubin AB nJy zeropoint
        print(f'✓ Loaded: shape={image.shape}')
        print(f'  Background std: {np.std(image):.4f} nJy')
        print(f'  Zeropoint: {ZEROPOINT} (Rubin nJy)')
    except:
        ZEROPOINT = 27.0
        print(f'  Could not get PhotoCalib, using default zp={ZEROPOINT}')
else:
    np.random.seed(42)
    image = np.random.normal(loc=100, scale=15, size=(2000, 2000)).astype(np.float32)
    exposure = None
    ZEROPOINT = 27.0
    print('Using mock image (no Butler connection)')

---
## 2b. Extract and Inspect the PSF from the Coadd

The deepCoadd carries a **spatially-varying PSF model** (a `CoaddPsf`).
We evaluate it at specific pixel positions using:
```python
psf_kernel = exposure.getPsf().computeImage(Point2D(x, y))
```
This is critical: injected clusters must be **convolved with the local PSF**
so they look like real detections.

In [ ]:
# ============================================================
# Extract the PSF at the image center
# ============================================================
if exposure is not None:
    cy, cx = image.shape[0] // 2, image.shape[1] // 2
    center_pos = Point2D(float(cx), float(cy))

    psf_kernel = get_psf_from_coadd(exposure, center_pos)
    psf_fwhm   = get_psf_fwhm_from_coadd(exposure, center_pos)
    psf_info   = get_psf_size_from_coadd(exposure, center_pos)

    print(f'PSF at center ({cx}, {cy}):')
    print(f'  Kernel shape  : {psf_kernel.shape}')
    print(f'  FWHM          : {psf_fwhm:.3f} pixels = {psf_fwhm * 0.2:.3f} arcsec')
    print(f'  Sigma         : {psf_info["sigma_px"]:.3f} pixels')
    print(f'  Ellipticity   : {psf_info["ellipticity"]:.4f}')
    print(f'  Moments (Ixx, Iyy, Ixy): ({psf_info["ixx"]:.3f}, {psf_info["iyy"]:.3f}, {psf_info["ixy"]:.3f})')
    print(f'  Kernel sum    : {psf_kernel.sum():.6f} (should be ~1.0)')
else:
    from scipy.ndimage import gaussian_filter
    psf_fwhm = 3.5
    psf_sigma = psf_fwhm / 2.355
    size = 41
    psf_kernel = np.zeros((size, size))
    psf_kernel[size//2, size//2] = 1.0
    psf_kernel = gaussian_filter(psf_kernel, sigma=psf_sigma)
    psf_kernel /= psf_kernel.sum()
    psf_info = {'fwhm_px': psf_fwhm, 'fwhm_arcsec': psf_fwhm*0.2,
                'sigma_px': psf_sigma, 'ellipticity': 0.0, 'kernel_shape': psf_kernel.shape,
                'ixx': psf_sigma**2, 'iyy': psf_sigma**2, 'ixy': 0.0}
    print(f'Mock Gaussian PSF: FWHM={psf_fwhm:.1f} px, kernel={psf_kernel.shape}')

In [ ]:
# ============================================================
# Visualize the PSF kernel
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 2D image of the PSF
ax = axes[0]
im = ax.imshow(psf_kernel, cmap='inferno', origin='lower')
ax.set_title(f'PSF Kernel ({psf_kernel.shape[0]}×{psf_kernel.shape[1]})')
ax.set_xlabel('pixels'); ax.set_ylabel('pixels')
plt.colorbar(im, ax=ax, shrink=0.8)

# Log-scale to see wings
ax = axes[1]
psf_pos = np.clip(psf_kernel, 1e-10, None)
im = ax.imshow(psf_pos, cmap='inferno', origin='lower',
               norm=LogNorm(vmin=psf_pos[psf_pos > 0].min(), vmax=psf_pos.max()))
ax.set_title('PSF (log scale — shows wings)')
ax.set_xlabel('pixels'); ax.set_ylabel('pixels')
plt.colorbar(im, ax=ax, shrink=0.8)

# Radial profile
ax = axes[2]
cy_k, cx_k = psf_kernel.shape[0]//2, psf_kernel.shape[1]//2
yy, xx = np.mgrid[:psf_kernel.shape[0], :psf_kernel.shape[1]]
rr = np.sqrt((xx - cx_k)**2 + (yy - cy_k)**2)
r_max = min(cx_k, cy_k)
r_bins = np.arange(0, r_max + 1, 0.5)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
radial_prof = np.array([psf_kernel[(rr >= r_bins[i]) & (rr < r_bins[i+1])].mean()
                        for i in range(len(r_bins)-1)])
ax.semilogy(r_centers, radial_prof, 'k-', lw=2)
ax.axvline(psf_fwhm / 2, color='red', ls='--', label=f'HWHM = {psf_fwhm/2:.2f} px')
ax.set_xlabel('Radius (pixels)')
ax.set_ylabel('Mean PSF value')
ax.set_title(f'Radial Profile — FWHM = {psf_fwhm:.2f} px ({psf_fwhm*0.2:.2f}")')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle(f'PSF from deepCoadd — {DATA_ID}', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Check PSF spatial variation across the coadd
# ============================================================
if exposure is not None:
    psf_samples = sample_psf_across_image(exposure, n_samples=9)

    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for idx, (ax, sample) in enumerate(zip(axes.flat, psf_samples)):
        ax.imshow(sample['psf_array'], cmap='inferno', origin='lower')
        ax.set_title(f"({sample['x']:.0f}, {sample['y']:.0f})\nFWHM={sample['fwhm_px']:.2f} px",
                     fontsize=9)
        ax.tick_params(labelsize=6)
    for ax in axes.flat[len(psf_samples):]:
        ax.axis('off')

    fwhm_vals = [s['fwhm_px'] for s in psf_samples]
    plt.suptitle(f'PSF across coadd — FWHM range: [{min(fwhm_vals):.2f}, {max(fwhm_vals):.2f}] px\n'
                 f'Variation: {max(fwhm_vals)-min(fwhm_vals):.3f} px = {(max(fwhm_vals)-min(fwhm_vals))*0.2:.3f} arcsec',
                 fontsize=12)
    plt.tight_layout(); plt.show()
else:
    print('Skipping spatial variation check (no exposure)')

In [ ]:
# ============================================================
# Demo: what PSF convolution does to a synthetic cluster
# ============================================================
from src.light_profiles import PlummerProfile

# Create a test Plummer profile
test_flux = mag_to_flux(22.0, zero_point=ZEROPOINT)
test_r_half = 8.0  # pixels

profile = PlummerProfile(total_flux=test_flux, r_half=test_r_half)
stamp_size = int(test_r_half * 10)
if stamp_size % 2 == 0: stamp_size += 1
yy, xx = np.mgrid[:stamp_size, :stamp_size]
cx_s, cy_s = stamp_size // 2, stamp_size // 2
raw_stamp = profile.surface_brightness(xx - cx_s, yy - cy_s)

# Convolve with real PSF
convolved_stamp = convolve_with_psf(raw_stamp, psf_kernel)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

vmax = max(raw_stamp.max(), convolved_stamp.max())
norm_kw = dict(cmap='inferno', origin='lower', vmin=0, vmax=vmax*0.5)

axes[0].imshow(raw_stamp, **norm_kw)
axes[0].set_title(f'Raw Plummer (mag=22, r½={test_r_half} px)')

axes[1].imshow(convolved_stamp, **norm_kw)
axes[1].set_title(f'After PSF convolution (FWHM={psf_fwhm:.1f} px)')

# Radial comparison
rr_stamp = np.sqrt((xx - cx_s)**2 + (yy - cy_s)**2).ravel()
order = np.argsort(rr_stamp)
axes[2].plot(rr_stamp[order], raw_stamp.ravel()[order], '.', ms=1, alpha=0.3, label='Raw')
axes[2].plot(rr_stamp[order], convolved_stamp.ravel()[order], '.', ms=1, alpha=0.3, label='PSF-convolved')
axes[2].set_xlabel('Radius (px)'); axes[2].set_ylabel('Surface brightness')
axes[2].set_title('Radial comparison')
axes[2].legend(); axes[2].set_yscale('log')
axes[2].set_ylim(bottom=max(1e-4, convolved_stamp[convolved_stamp > 0].min() * 0.1))

plt.suptitle('Effect of PSF on injected cluster profile', fontsize=13)
plt.tight_layout(); plt.show()

print(f'Raw peak     : {raw_stamp.max():.4f}')
print(f'Convolved peak: {convolved_stamp.max():.4f}')
print(f'Flux preserved: {convolved_stamp.sum() / raw_stamp.sum():.4f} (should be ~1.0)')

In [ ]:
# MOCK IMAGE - uncomment if Butler fails
# np.random.seed(42)
# image = np.random.normal(loc=100, scale=15, size=(2000, 2000)).astype(np.float32)
# exposure = None
# ZEROPOINT = 27.0

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
vmin, vmax = np.percentile(image, [1, 99])
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title(f"Coadd: {DATA_ID}  |  shape={image.shape}")
ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')
plt.tight_layout(); plt.show()

---
## 3. Configuration

Unlike the Rubin tutorial which injects **point sources (stars)**,
we inject **extended sources** using analytical 2D profiles.

| Method | What it is | Rubin tutorial equivalent |
|--------|-----------|---------------------------|
| `smooth` | One 2D surface brightness profile per cluster | Extended `galsim.Sersic` source |
| `discrete` | N individual point-source stars | Same as Rubin tutorial stars, but placed by IMF |

In [ ]:
# ============================================================
# CHANGE THESE TO CONFIGURE YOUR INJECTION
# ============================================================

METHOD       = 'smooth'
PROFILE_TYPE = 'plummer'  # 'plummer', 'king', 'eff', 'sersic'
N_CLUSTERS   = 20

MAG_MIN    = 19.0
MAG_MAX    = 25.0
R_HALF_MIN = 3.0   # pixels (0.6 arcsec at Rubin pixel scale)
R_HALF_MAX = 25.0  # pixels (5.0 arcsec)

# ---- PSF convolution ----
CONVOLVE_PSF = True
# Use spatially varying PSF (evaluated at each cluster position)
# Set False to use a single PSF from the image center (faster)
SPATIALLY_VARYING_PSF = True

# ---- Multi-band ----
MULTIBAND  = False
BANDS      = ['g', 'r', 'i']
MAG_RANGES = {'g': (19.5, 26.0), 'r': (19.0, 25.5), 'i': (18.5, 25.0)}
BAND       = 'i'

# ---- Discrete only ----
N_STARS_MIN        = 100
N_STARS_MAX        = 500
N_STARS_FIXED      = None
IMF                = 'kroupa'
AGE_GYR_MIN        = 0.5
AGE_GYR_MAX        = 10.0
AGE_GYR_FIXED      = None
METALLICITY_MIN    = 0.001
METALLICITY_MAX    = 0.04
METALLICITY_FIXED  = None
DISTANCE_PC_MIN    = 5000
DISTANCE_PC_MAX    = 50000
DISTANCE_PC_FIXED  = None
BINARY_FRACTION    = 0.3

ADD_NOISE    = True
EDGE_BUFFER  = 150
SEED         = 42

bg_std = np.std(image)
print(f'Method    : {METHOD} ({"one 2D profile" if METHOD=="smooth" else "N individual stars"})')
print(f'Profile   : {PROFILE_TYPE}')
print(f'N clusters: {N_CLUSTERS}')
print(f'PSF conv  : {CONVOLVE_PSF} (spatially varying: {SPATIALLY_VARYING_PSF})')
print(f'PSF FWHM  : {psf_fwhm:.2f} px = {psf_fwhm*0.2:.2f} arcsec')
print(f'Multiband : {MULTIBAND} → {BANDS if MULTIBAND else [BAND]}')
print(f'BG std    : {bg_std:.4f}')

# Quick zeropoint sanity check
test_flux = mag_to_flux(MAG_MIN, zero_point=ZEROPOINT)
print(f'\nFlux check at MAG_MIN={MAG_MIN}: {test_flux:.4f} ({"OK" if test_flux > bg_std else "WARNING: too faint!"})')
if test_flux < bg_std:
    print('  ⚠️  Try MAG_MIN=16-18, or set ZEROPOINT=31.4')

---
## 4. Create Injection Catalog

In [ ]:
if MULTIBAND:
    catalog = create_multiband_catalog(
        n_clusters=N_CLUSTERS, image_shape=image.shape,
        bands=BANDS, mag_ranges=MAG_RANGES,
        r_half_range=(R_HALF_MIN, R_HALF_MAX),
        profile_type=PROFILE_TYPE, method=METHOD,
        edge_buffer=EDGE_BUFFER, exposure=exposure, seed=SEED
    )
else:
    catalog = create_injection_catalog(
        n_clusters=N_CLUSTERS, image_shape=image.shape,
        mag_range=(MAG_MIN, MAG_MAX),
        r_half_range=(R_HALF_MIN, R_HALF_MAX),
        profile_type=PROFILE_TYPE, method=METHOD,
        n_stars_range=(N_STARS_MIN, N_STARS_MAX),
        n_stars_fixed=N_STARS_FIXED,
        imf=IMF,
        age_gyr_range=(AGE_GYR_MIN, AGE_GYR_MAX),
        age_gyr_fixed=AGE_GYR_FIXED,
        metallicity_range=(METALLICITY_MIN, METALLICITY_MAX),
        metallicity_fixed=METALLICITY_FIXED,
        distance_pc_range=(DISTANCE_PC_MIN, DISTANCE_PC_MAX),
        distance_pc_fixed=DISTANCE_PC_FIXED,
        binary_fraction=BINARY_FRACTION,
        edge_buffer=EDGE_BUFFER, exposure=exposure, seed=SEED
    )

print(f'✓ {len(catalog)} clusters')
print(f'{"ID":>4} {"X":>6} {"Y":>6} {"Mag":>6} {"r_half":>7}')
print('-'*35)
for e in catalog[:10]:
    print(f"{e['id']:4d} {e['x']:6.0f} {e['y']:6.0f} {e['magnitude']:6.1f} {e['r_half']:7.1f}")

---
## 5. Run Injection (with PSF convolution)

In [ ]:
from src.inject import inject_from_catalog, inject_multiband

# ---- Build PSF kwargs based on configuration ----
psf_kwargs = {}
if CONVOLVE_PSF:
    if SPATIALLY_VARYING_PSF and exposure is not None:
        psf_kwargs['psf_mode'] = 'spatially_varying'
        print(f'PSF mode: spatially_varying (evaluated at each cluster position)')
    else:
        psf_kwargs['psf_mode'] = 'fixed'
        psf_kwargs['psf_kernel'] = psf_kernel
        print(f'PSF mode: fixed (center kernel, FWHM={psf_fwhm:.2f} px)')
else:
    psf_kwargs['psf_mode'] = 'none'
    print('PSF mode: none (raw analytical profiles, no convolution)')

if MULTIBAND:
    print('\nLoading exposures for all bands...')
    exposures = {}
    images_dict = {}
    for band in BANDS:
        try:
            exp = butler.get('deepCoadd', dataId={**DATA_ID, 'band': band})
            exposures[band]    = exp
            images_dict[band]  = exp.image.array.copy()
            print(f'  ✓ {band}-band loaded: shape={images_dict[band].shape}')
        except Exception as e:
            print(f'  ✗ {band}-band failed: {e}')

    print(f'\nInjecting into {list(images_dict.keys())} bands...')
    injected_images, injection_info_multiband = inject_multiband(
        images_dict, catalog,
        exposures=exposures,
        add_noise=ADD_NOISE,
        pixel_scale=0.2,
        save_stamps=True,
        **psf_kwargs
    )
    default_band = BANDS[-1]
    injected_image = injected_images[default_band]
    injection_info = injection_info_multiband[default_band]
    image_for_diff = images_dict[default_band]
    print(f'\nUsing {default_band}-band for diagnostics below')

else:
    print(f'\nInjecting {len(catalog)} {METHOD} clusters ({PROFILE_TYPE} profile)...')
    injected_image, injection_info = inject_from_catalog(
        image, catalog,
        exposure=exposure,
        add_noise=ADD_NOISE,
        pixel_scale=0.2,
        save_stamps=True,
        **psf_kwargs
    )
    image_for_diff = image

total_flux = sum(e['total_flux_injected'] for e in injection_info)
n_convolved = sum(1 for e in injection_info if e.get('psf_convolved', False))
print(f'\n✓ Injection complete')
print(f'  Clusters injected : {len(injection_info)}')
print(f'  PSF-convolved     : {n_convolved}/{len(injection_info)}')
print(f'  Total flux        : {total_flux:.2f}')

---
## 6. Visualize Injection Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = np.percentile(image_for_diff, [1, 99])

axes[0].imshow(image_for_diff, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Original')

axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
for c in catalog:
    axes[1].scatter(c['x'], c['y'], s=50, facecolors='none', edgecolors='red', lw=1)
axes[1].set_title(f'With {len(catalog)} Injected Clusters')

diff = injected_image - image_for_diff
diff_pos = diff.copy().astype(float)
diff_pos[diff_pos <= 0] = np.nan
if np.nanmax(diff_pos) > 0:
    im = axes[2].imshow(diff_pos, cmap='hot', origin='lower',
                         norm=LogNorm(vmin=np.nanpercentile(diff_pos[np.isfinite(diff_pos)], 5),
                                      vmax=np.nanmax(diff_pos)))
    plt.colorbar(im, ax=axes[2])
axes[2].set_title('Injected Clusters Only (difference)')

plt.suptitle(f'Injection: {METHOD} {PROFILE_TYPE}, PSF convolved={CONVOLVE_PSF}', fontsize=13)
plt.tight_layout(); plt.show()

## 7. Verify PSF Effect on Injected Stamps

In [ ]:
# Show postage stamps of injected clusters to verify PSF convolution
n_show = min(6, len(injection_info))
fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))
if n_show == 1:
    axes = axes.reshape(2, 1)

for i in range(n_show):
    info = injection_info[i]
    cx_i, cy_i = int(info['x']), int(info['y'])
    r = max(int(info['r_half'] * 5), 20)

    y0, y1 = max(0, cy_i - r), min(injected_image.shape[0], cy_i + r)
    x0, x1 = max(0, cx_i - r), min(injected_image.shape[1], cx_i + r)
    stamp_inj  = injected_image[y0:y1, x0:x1]
    stamp_orig = image_for_diff[y0:y1, x0:x1]

    v1, v2 = np.percentile(stamp_inj, [1, 99.5])
    axes[0, i].imshow(stamp_orig, cmap='gray', origin='lower', vmin=v1, vmax=v2)
    axes[0, i].set_title(f'Original\nmag={info["magnitude"]:.1f}', fontsize=9)

    axes[1, i].imshow(stamp_inj, cmap='gray', origin='lower', vmin=v1, vmax=v2)
    psf_tag = '✓PSF' if info.get('psf_convolved', False) else '✗no PSF'
    axes[1, i].set_title(f'Injected (r½={info["r_half"]:.1f}px) {psf_tag}', fontsize=9)

    for row in range(2):
        axes[row, i].plot(cx_i - x0, cy_i - y0, 'r+', ms=10, mew=1.5)

axes[0, 0].set_ylabel('Before injection')
axes[1, 0].set_ylabel('After injection')
plt.suptitle(f'Injected cluster stamps — PSF mode: {psf_kwargs.get("psf_mode", "none")}', fontsize=13)
plt.tight_layout(); plt.show()

## 8. Save Results

In [ ]:
# Save injection catalog and info as JSON
output_dir = os.path.join(PIPELINE_PATH, 'results')
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
catalog_file = os.path.join(output_dir, f'injection_catalog_{timestamp}.json')
info_file    = os.path.join(output_dir, f'injection_info_{timestamp}.json')

def make_serializable(obj):
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, (np.float32, np.float64)): return float(obj)
    if isinstance(obj, (np.int32, np.int64)): return int(obj)
    return obj

catalog_out = [{k: make_serializable(v) for k, v in entry.items()} for entry in catalog]
info_out = [{k: make_serializable(v) for k, v in entry.items()
             if not isinstance(v, np.ndarray)} for entry in injection_info]

with open(catalog_file, 'w') as f:
    json.dump(catalog_out, f, indent=2)
with open(info_file, 'w') as f:
    json.dump(info_out, f, indent=2)

print(f'✓ Catalog saved: {catalog_file}')
print(f'✓ Info saved:    {info_file}')
print(f'  {len(catalog_out)} clusters, PSF mode={psf_kwargs.get("psf_mode", "none")}')

## Quick Reference: Configuration Options

### Cluster Methods
- `'smooth'`: Extended source with analytical profile (fast, good for distant clusters)
- `'discrete'`: Individual stars with IMF and spatial distribution (realistic, resolved clusters)

### Profile Types
- `'plummer'`: Simple analytical profile, good for many clusters
- `'king'`: Tidally truncated, good for globular clusters
- `'eff'`: Power-law profile, good for young clusters
- `'sersic'`: Generalized profile (n=1: exponential, n=4: de Vaucouleurs)

### IMF Options (for discrete stars)
- `'kroupa'`: Kroupa (2001) IMF
- `'chabrier'`: Chabrier (2003) IMF
- `'salpeter'`: Salpeter (1955) IMF

---
## 10. Summary Diagnostic Panel

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

mags   = [i['magnitude'] for i in injection_info]
sizes  = [i['r_half']    for i in injection_info]
fluxes = [i['total_flux_injected'] for i in injection_info]
peak_snrs = [i['peak_brightness'] / bg_std for i in injection_info]

vmin_d, vmax_d = np.percentile(image_for_diff, [1, 99])

# Original
axes[0,0].imshow(image_for_diff, cmap='gray', origin='lower', vmin=vmin_d, vmax=vmax_d)
axes[0,0].set_title('Original Coadd')

# Injected + circles
axes[0,1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin_d, vmax=vmax_d)
for info in injection_info:
    col = plt.cm.plasma(np.clip((info['magnitude']-MAG_MIN)/(MAG_MAX-MAG_MIN), 0, 1))
    axes[0,1].scatter(info['x'], info['y'], s=60, facecolors='none', edgecolors=col, lw=1.5)
axes[0,1].set_title(f'Injected ({len(injection_info)} clusters)')

# Difference
diff = injected_image - image_for_diff
if diff.max() > 0:
    im = axes[0,2].imshow(diff, cmap='hot', origin='lower',
                          norm=LogNorm(vmin=max(0.01,np.percentile(diff[diff>0],5) if np.any(diff>0) else 0.01),
                                       vmax=diff.max()))
    plt.colorbar(im, ax=axes[0,2], label='Injected flux')
axes[0,2].set_title('Difference (clusters only)')

# Magnitude hist
axes[1,0].hist(mags, bins=15, edgecolor='black', color='steelblue')
axes[1,0].axvline(np.median(mags), color='red', ls='--', label=f'median={np.median(mags):.1f}')
axes[1,0].set_xlabel('Magnitude'); axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Magnitude Distribution'); axes[1,0].legend()

# Size hist
axes[1,1].hist(sizes, bins=15, edgecolor='black', color='darkorange')
axes[1,1].axvline(np.median(sizes), color='red', ls='--', label=f'median={np.median(sizes):.1f}px')
axes[1,1].set_xlabel('r_half (pixels)'); axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Size Distribution'); axes[1,1].legend()

# Flux vs mag
sc = axes[1,2].scatter(mags, np.log10(np.array(fluxes)+1),
                        c=sizes, cmap='plasma', s=40, alpha=0.8)
plt.colorbar(sc, ax=axes[1,2], label='r_half (px)')
axes[1,2].set_xlabel('Magnitude'); axes[1,2].set_ylabel('log10(Flux Injected)')
axes[1,2].set_title('Flux vs Magnitude (colored by size)')
textstr = (f'N injected : {len(injection_info)}\n'
           f'Mag range  : {min(mags):.1f} – {max(mags):.1f}\n'
           f'r½ range   : {min(sizes):.1f} – {max(sizes):.1f} px\n'
           f'Median S/N : {np.median(peak_snrs):.1f}\n'
           f'PSF FWHM   : {psf_fwhm:.2f} px\n'
           f'PSF mode   : {psf_kwargs.get("psf_mode", "none")}\n'
           f'BG std     : {bg_std:.4f}')
axes[1,2].text(0.03, 0.97, textstr, transform=axes[1,2].transAxes,
               fontsize=8, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

for ax in axes.flat:
    if not ax.get_xlabel():
        ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')

plt.suptitle(f'Injection Summary — {METHOD} {PROFILE_TYPE}, {DATA_ID["band"]}-band, PSF={psf_kwargs.get("psf_mode","none")}', fontsize=14)
plt.tight_layout(); plt.show()

---
## 11. Detection Pipeline

Implements detection methods from three papers:

| Method | Reference |
|--------|----------|
| Matched-filter + background subtraction | [Saifollahi+2025](https://scixplorer.org/abs/2025A%26A...697A..10S/abstract) (Euclid injection) |
| Multiple Concentration Index (MCI) | [Thilker+2022](https://scixplorer.org/abs/2022MNRAS.509.4094T/abstract) (PHANGS-HST) |
| Multi-scale completeness framework | [Hannon+2024](https://scixplorer.org/abs/2024AJ....168...38H/abstract) |

### Pipeline flow
```
injected_image
    ↓
estimate_background()      # sliding box median + sigma clip
    ↓
matched_filter_detect()    # convolve with cluster kernel, threshold
    ↓
extract_sources()          # centroid, flux, shape
    ↓
compute_mci()              # inner/outer concentration index
    ↓
select_cluster_candidates() # MCI + S/N + shape cuts
    ↓
ClusterRetrieval()         # match to injections
```

In [ ]:
from src.detection import (
    run_cluster_detection,
    estimate_background,
    matched_filter_detect,
    extract_sources,
    compute_mci,
    select_cluster_candidates,
    plot_detection_diagnostic,
    plot_mci_diagram
)

print('✓ Detection module loaded')

In [ ]:
# ============================================================
# DETECTION CONFIGURATION
# Use the PSF FWHM measured from the coadd (Section 2b)
# ============================================================

PSF_FWHM_PX = psf_fwhm
print(f'PSF FWHM (from coadd): {PSF_FWHM_PX:.2f} pixels = {PSF_FWHM_PX*0.2:.2f} arcsec')

DETECT_THRESH = 3.0

DETECT_SCALES = [max(2.0, PSF_FWHM_PX*0.5),
                  PSF_FWHM_PX,
                  PSF_FWHM_PX*2,
                  PSF_FWHM_PX*4]

MCI_MAX = 0.9
SNR_MIN = 3.0
USE_MULTISCALE = True
USE_MCI = True

print(f'\nDetection config:')
print(f'  Threshold  : {DETECT_THRESH}σ')
print(f'  Scales     : {[f"{s:.1f}" for s in DETECT_SCALES]} px')
print(f'  MCI cut    : {USE_MCI} (max={MCI_MAX})')
print(f'  Multi-scale: {USE_MULTISCALE}')

In [ ]:
# Run the detection pipeline
detections = run_cluster_detection(
    injected_image,
    psf_fwhm=PSF_FWHM_PX,
    threshold_sigma=DETECT_THRESH,
    r_half_scales=DETECT_SCALES,
    mci_max=MCI_MAX,
    snr_min=SNR_MIN,
    r_half_min=1.0,
    ellipticity_max=0.6,
    pixel_scale=0.2,
    use_multiscale=USE_MULTISCALE,
    use_mci=USE_MCI,
    verbose=True
)

print(f'\n✓ Detected {len(detections)} cluster candidates')
if detections:
    snrs = [d['snr'] for d in detections]
    rhs  = [d.get('r_half', 0) for d in detections]
    print(f'  S/N range  : [{min(snrs):.1f}, {max(snrs):.1f}]')
    print(f'  r_half range: [{min(rhs):.1f}, {max(rhs):.1f}] px')

In [ ]:
# ---- Step-by-step diagnostic ----
background, rms = estimate_background(injected_image, box_size=64)

r_diag = DETECT_SCALES[1]
snr_map_diag, det_map_diag = matched_filter_detect(
    injected_image, background, rms,
    r_half_pixels=r_diag,
    threshold_sigma=DETECT_THRESH
)

fig = plot_detection_diagnostic(
    injected_image, background, rms,
    snr_map_diag, detections,
    injection_info=injection_info
)
plt.show()

In [ ]:
# ---- MCI Diagram (Thilker+2022 style) ----
if USE_MCI and detections and 'inner_mci' in detections[0]:
    fig = plot_mci_diagram(detections, injection_info=injection_info)
    plt.show()
else:
    print('MCI not computed (USE_MCI=False or no detections)')

In [ ]:
# ---- Threshold sensitivity test ----
from src.retrieval import ClusterRetrieval

thresholds  = [2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
n_detected  = []
completeness_at_thresh = []

for thresh in thresholds:
    dets = run_cluster_detection(
        injected_image,
        psf_fwhm=PSF_FWHM_PX,
        threshold_sigma=thresh,
        r_half_scales=DETECT_SCALES,
        mci_max=MCI_MAX, snr_min=thresh,
        use_multiscale=USE_MULTISCALE,
        use_mci=USE_MCI,
        verbose=False
    )
    ret = ClusterRetrieval(injection_info, dets)
    ret.match_detections(match_radius=5.0)
    stats = ret.get_summary_statistics()
    n_detected.append(len(dets))
    completeness_at_thresh.append(stats['overall_completeness'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(thresholds, n_detected, 'o-', lw=2)
ax.set_xlabel('Detection Threshold (σ)')
ax.set_ylabel('N detections')
ax.set_title('Detections vs Threshold')
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(thresholds, completeness_at_thresh, 'o-', lw=2, color='orange')
ax.axhline(0.5, color='red', ls='--', label='50%')
ax.set_xlabel('Detection Threshold (σ)')
ax.set_ylabel('Overall Completeness')
ax.set_title('Completeness vs Threshold')
ax.set_ylim(0, 1.05)
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Detection Threshold Sensitivity (Hannon+2024 framework)', fontsize=13)
plt.tight_layout(); plt.show()

print('Threshold | N detected | Completeness')
for t, n, c in zip(thresholds, n_detected, completeness_at_thresh):
    print(f'  {t:.1f}σ    |  {n:4d}      |  {c:.1%}')

---
## 12. Match Injections to Detections

...existing content...